In [ ]:
import duckdb
con = duckdb.connect("../data/processed/complaints.duckdb")

In [2]:
#Are complaints going up or down?
con.execute("""
    WITH monthly AS (
        SELECT month, COUNT(*) AS complaints
        FROM complaints
        WHERE month BETWEEN '2024-01' AND '2026-10'
        GROUP BY month
    )
    SELECT month,
           complaints,
           LAG(complaints) OVER (ORDER BY month) AS previous_month,
           ROUND((complaints - LAG(complaints) OVER (ORDER BY month)) * 100.0
                 / LAG(complaints) OVER (ORDER BY month), 1) AS pct_change
    FROM monthly
    ORDER BY month
""").df()

,month,complaints,previous_month,pct_change
0,2024-01,625,<NA>,NaN
1,2024-02,795,625,27.2
2,2024-03,943,795,18.6
3,2024-04,880,943,-6.7
4,2024-05,932,880,5.9
5,2024-06,930,932,-0.2
6,2024-07,986,930,6.0
7,2024-08,1280,986,29.8
8,2024-09,1279,1280,-0.1
9,2024-10,1735,1279,35.7


In [3]:
#Who drives credit reporting complaints?
con.execute("""
    SELECT company,
           COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_credit_reporting
    FROM complaints
    WHERE is_credit_reporting
    GROUP BY company
    ORDER BY complaints DESC
    LIMIT 5
""").df()

,company,complaints,pct_of_credit_reporting
0,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",30460,34.4
1,"EQUIFAX, INC.",28237,31.9
2,EXPERIAN INFORMATION SOLUTIONS INC.,26189,29.6
3,LEXISNEXIS,436,0.5
4,"CBC COMPANIES, INC.",326,0.4


In [4]:
#What are the top problems in credit reporting
con.execute("""
    SELECT issue, COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM complaints
    WHERE is_credit_reporting
    GROUP BY issue
    ORDER BY complaints DESC
    LIMIT 5
""").df()

,issue,complaints,pct
0,Incorrect information on your report,50946,57.6
1,Problem with a company's investigation into an...,20344,23.0
2,Improper use of your report,16315,18.4
3,Problem with a company's investigation into an...,330,0.4
4,Unable to get your credit report or credit score,295,0.3


In [5]:
#Outside credit reporting, which products have the most complaints
con.execute("""
    SELECT product, COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_non_credit
    FROM complaints
    WHERE NOT is_credit_reporting
    GROUP BY product
    ORDER BY complaints DESC
""").df()

,product,complaints,pct_of_non_credit
0,Debt collection,5722,51.1
1,Credit card,1330,11.9
2,Checking or savings account,1297,11.6
3,"Money transfer, virtual currency, or money ser...",1114,9.9
4,Mortgage,448,4.0
5,Vehicle loan or lease,434,3.9
6,Student loan,306,2.7
7,"Payday loan, title loan, personal loan, or adv...",275,2.5
8,Prepaid card,178,1.6
9,Debt or credit management,95,0.8


In [6]:
#Top 3 issues with each product
con.execute("""
    WITH counts AS (
        SELECT product, issue, COUNT(*) AS complaints
        FROM complaints
        WHERE NOT is_credit_reporting
        GROUP BY product, issue
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product ORDER BY complaints DESC) AS rank_in_product
        FROM counts
    )
    SELECT product, rank_in_product, issue, complaints
    FROM ranked
    WHERE rank_in_product <= 3
    ORDER BY product, rank_in_product
""").df()

,product,rank_in_product,issue,complaints
0,Checking or savings account,1,Managing an account,708
1,Checking or savings account,2,Problem with a lender or other company chargin...,216
2,Checking or savings account,3,Closing an account,148
3,Credit card,1,Problem with a purchase shown on your statement,329
4,Credit card,2,Problem when making payments,170
5,Credit card,3,Fees or interest,153
6,Debt collection,1,Attempts to collect debt not owed,2897
7,Debt collection,2,False statements or representation,914
8,Debt collection,3,Written notification about debt,882
9,Debt or credit management,1,Charged upfront or unexpected fees,25


In [3]:
#Which companies have the most untimely responses
con.execute("""
    SELECT company,
           COUNT(*) AS total_complaints,
           SUM(CASE WHEN timely_response = 'No' THEN 1 ELSE 0 END) AS untimely,
           ROUND(SUM(CASE WHEN timely_response = 'No' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_untimely
    FROM complaints
    GROUP BY company
    HAVING COUNT(*) >= 100
    ORDER BY pct_untimely DESC
    LIMIT 10
""").df()

,company,total_complaints,untimely,pct_untimely
0,MOHELA,131,67.0,51.15
1,"CCS FINANCIAL SERVICES, INC.",107,18.0,16.82
2,SYNCHRONY FINANCIAL,268,7.0,2.61
3,"PORTFOLIO RECOVERY ASSOCIATES, LLC",343,8.0,2.33
4,"BLOCK, INC.",899,15.0,1.67
5,LEXISNEXIS,436,4.0,0.92
6,AMERICAN EXPRESS COMPANY,142,1.0,0.70
7,CREDIT ACCEPTANCE CORPORATION,140,0.0,0.00
8,"EQUIFAX, INC.",28922,0.0,0.00
9,"BANK OF AMERICA, NATIONAL ASSOCIATION",230,0.0,0.00


In [4]:
#Which companies have the most complaints still in progress
con.execute("""
    SELECT company,
           COUNT(*) AS total_complaints,
           SUM(CASE WHEN company_response_to_consumer = 'In progress' THEN 1 ELSE 0 END) AS in_progress
    FROM complaints
    GROUP BY company
    HAVING COUNT(*) >= 100
    ORDER BY in_progress DESC
    LIMIT 10
""").df()

,company,total_complaints,in_progress
0,EXPERIAN INFORMATION SOLUTIONS INC.,26646,2664.0
1,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",31123,1577.0
2,"EQUIFAX, INC.",28922,183.0
3,SYNCHRONY FINANCIAL,268,14.0
4,MOHELA,131,9.0
5,CHIME FINANCIAL INC,285,9.0
6,"KRIYA CAPITAL, LLC",145,9.0
7,"BANK OF AMERICA, NATIONAL ASSOCIATION",230,8.0
8,NAVY FEDERAL CREDIT UNION,154,4.0
9,"PORTFOLIO RECOVERY ASSOCIATES, LLC",343,4.0


“In progress” complaints are probably recent ones that haven’t been answered yet, so check whether they’re clustered in the latest months before you call a company slow.

In [5]:
#How do outcomes differ by product?
con.execute("""
    SELECT product,
           COUNT(*) AS closed_complaints,
           ROUND(SUM(CASE WHEN company_response_to_consumer = 'Closed with monetary relief' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_monetary_relief,
           ROUND(SUM(CASE WHEN company_response_to_consumer = 'Closed with non-monetary relief' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_non_monetary,
           ROUND(SUM(CASE WHEN company_response_to_consumer = 'Closed with explanation' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_explanation
    FROM complaints
    WHERE company_response_to_consumer LIKE 'Closed%'
    GROUP BY product
    HAVING COUNT(*) >= 100
    ORDER BY closed_complaints DESC
""").df()

,product,closed_complaints,pct_monetary_relief,pct_non_monetary,pct_explanation
0,Credit reporting or other personal consumer re...,84040,0.0,38.9,61.0
1,Debt collection,5567,0.1,29.0,70.9
2,Credit card,1300,11.3,16.9,71.8
3,Checking or savings account,1281,10.5,5.0,84.5
4,"Money transfer, virtual currency, or money ser...",1109,1.9,2.7,95.4
5,Mortgage,435,2.8,4.1,93.1
6,Vehicle loan or lease,425,0.9,4.5,94.6
7,"Payday loan, title loan, personal loan, or adv...",269,2.6,4.1,93.3
8,Student loan,250,0.8,3.2,96.0
9,Prepaid card,176,11.9,13.1,75.0


In [6]:
#Which Arkansas areas have the most complaints?
con.execute("""
    SELECT zip3,
           COUNT(*) AS complaints,
           SUM(CASE WHEN is_credit_reporting THEN 0 ELSE 1 END) AS non_credit_reporting
    FROM complaints
    GROUP BY zip3
    ORDER BY complaints DESC
    LIMIT 10
""").df()

,zip3,complaints,non_credit_reporting
0,722,18808,1832.0
1,721,16577,1758.0
2,720,15818,1942.0
3,723,12575,880.0
4,716,11084,806.0
5,727,7853,1285.0
6,724,4841,633.0
7,717,3333,363.0
8,718,3189,310.0
9,729,2049,422.0


In [7]:
#Servicemember and Older American complaints
con.execute("""
    SELECT tags, COUNT(*) AS complaints,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM complaints
    GROUP BY tags
    ORDER BY complaints DESC
""").df()

,tags,complaints,pct
0,None,96779,97.1
1,Servicemember,2218,2.2
2,Older American,542,0.5
3,"Older American, Servicemember",170,0.2


In [1]:
import os
print(os.listdir("../data/processed"))

['complaints.duckdb', 'monthly_summary.csv', 'complaints_clean.parquet', 'complaints_tableau.csv']


In [2]:
import pandas as pd
pd.read_csv("../data/processed/monthly_summary.csv").head()

,month,complaints,credit_reporting,other_products
0,2024-01,625,503,122
1,2024-02,795,653,142
2,2024-03,943,799,144
3,2024-04,880,756,124
4,2024-05,932,752,180
